# Sesión 03 - Variables aleatorias

Objetivo: transformar resultados aleatorios en variables discretas y continuas, y trabajar con PMF, PDF aproximada y CDF.


In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(7)
pd.set_option("display.precision", 4)


## 1. Una variable aleatoria discreta sobre dos dados

Definimos $X$ como la suma de dos dados.


In [ ]:
omega = np.array(list(product(range(1, 7), repeat=2)))
X = omega.sum(axis=1)

valores, conteos = np.unique(X, return_counts=True)
pmf = pd.DataFrame({"x": valores, "P(X=x)": conteos / conteos.sum()})
pmf["F(X<=x)"] = pmf["P(X=x)"].cumsum()
pmf


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(pmf["x"], pmf["P(X=x)"])
axes[0].set_title("PMF de X = suma de dos dados")
axes[0].set_xlabel("x")
axes[0].set_ylabel("probabilidad")

axes[1].step(pmf["x"], pmf["F(X<=x)"], where="post")
axes[1].set_title("CDF discreta")
axes[1].set_xlabel("x")
axes[1].set_ylabel("F(x)")
axes[1].set_ylim(0, 1.05)
plt.show()


## 2. Transformaciones de variables aleatorias

Del mismo experimento podemos derivar otra variable: $Y = 1\{X \ge 10\}$.


In [ ]:
Y = (X >= 10).astype(int)
pd.Series(Y).value_counts(normalize=True).sort_index().rename("P(Y=y)").to_frame()


## 3. Variable continua: tiempo de espera

Simulamos tiempos de espera exponenciales. En variables continuas no se pregunta por un punto exacto, sino por intervalos.


In [ ]:
tasa = 1 / 8  # espera media de 8 minutos
espera = rng.exponential(scale=1 / tasa, size=20_000)

p_entre_5_y_10_emp = np.mean((espera >= 5) & (espera <= 10))
p_entre_5_y_10_teo = stats.expon(scale=1 / tasa).cdf(10) - stats.expon(scale=1 / tasa).cdf(5)

print(f"P(5 <= T <= 10) empírica: {p_entre_5_y_10_emp:.4f}")
print(f"P(5 <= T <= 10) teórica:  {p_entre_5_y_10_teo:.4f}")
print(f"P(T = 8) en modelo continuo: 0")


In [ ]:
xs = np.linspace(0, 40, 300)
pdf = stats.expon(scale=1 / tasa).pdf(xs)
cdf = stats.expon(scale=1 / tasa).cdf(xs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(espera, bins=40, density=True, alpha=0.45, label="simulación")
axes[0].plot(xs, pdf, color="crimson", label="PDF teórica")
axes[0].set_title("PDF de tiempo de espera")
axes[0].legend()

axes[1].plot(xs, cdf, color="darkgreen")
axes[1].set_title("CDF de tiempo de espera")
axes[1].set_ylim(0, 1.02)
plt.show()


## 4. Estandarización

La transformación $Z=(X-\mu)/\sigma$ permite comparar variables en escalas distintas.


In [ ]:
z_espera = (espera - espera.mean()) / espera.std(ddof=0)
resumen = pd.Series(z_espera).describe(percentiles=[0.05, 0.5, 0.95])
resumen


## 5. Lectura de variables desde fuentes de datos

Si `data_sources/sales_data.csv` está disponible, este bloque identifica variables discretas, continuas, binarias y categóricas del dataset de demanda usado en el material original.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró data_sources/sales_data.csv. Se mantiene la práctica con datos sintéticos.")
else:
    ventas_df = pd.read_csv(DATA_DIR / "sales_data.csv", nrows=5_000, parse_dates=["Date"])
    catalogo = pd.DataFrame(
        {
            "columna": ventas_df.columns,
            "dtype": [str(ventas_df[c].dtype) for c in ventas_df.columns],
            "n_unicos": [ventas_df[c].nunique() for c in ventas_df.columns],
            "ejemplo": [ventas_df[c].dropna().iloc[0] for c in ventas_df.columns],
        }
    )
    catalogo["lectura_probabilistica"] = np.select(
        [
            catalogo["n_unicos"].eq(2),
            catalogo["dtype"].str.contains("int|float") & catalogo["n_unicos"].le(30),
            catalogo["dtype"].str.contains("int|float"),
            catalogo["dtype"].str.contains("datetime"),
        ],
        ["binaria", "discreta/conteo", "continua o discreta amplia", "índice temporal"],
        default="categórica",
    )
    display(catalogo)


## Práctica

Define una nueva variable aleatoria sobre el lanzamiento de dos dados: máximo, mínimo, diferencia absoluta o indicador de dobles. Construye su PMF y CDF.
